# Assignment: diffusing an image

In this assignment you will implement the forward and reverse diffusion process on a real
image. You take a photo, add Gaussian noise one step at a time until it dissolves into
static, then run the process backwards to reassemble it. Because you feed the **true**
$x_0$ into every reverse step, a correct implementation recovers the original almost exactly
— which makes this a clean way to check your formulas.

**What is given:** all imports, image loading, and display helpers (the cells marked
*GIVEN — run, don't edit*).

**What you write:** the $\beta$ schedule, the two `*_sample` functions, and the
forward/reverse loops. Each task has instructions and the torch commands you'll likely need.

A correct submission produces: a dissolving forward filmstrip, a reassembling reverse
filmstrip, and a final recovery RMSE near zero.

## torch quick reference

You already know numpy; torch is nearly identical. The commands you'll need:

| task | torch | numpy analog |
|------|-------|--------------|
| make a tensor from a list | `torch.tensor([...])` | `np.array([...])` |
| cumulative product along axis 0 | `torch.cumprod(x, 0)` | `np.cumprod(x)` |
| square root | `torch.sqrt(x)` | `np.sqrt(x)` |
| Gaussian noise shaped like `x` | `torch.randn_like(x)` | `np.random.randn(*x.shape)` |
| indexing & broadcasting | `x[t]`, `a*x + b` | same |
| a scalar zero tensor | `torch.tensor(0.0)` | `np.float32(0.0)` |

Indexing and arithmetic broadcast exactly like numpy. The main new one is
`torch.randn_like(x)`, which makes noise matching the shape **and** dtype of `x` in one call.

---
## GIVEN — load your image below and the run, don't edit these cells

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
## Download your image and open it here

img = Image.open('knightley.png').convert('RGB')
img_array = np.array(img).astype(np.float32) / 255.0
plt.imshow(img_array); plt.axis('off'); plt.show()
print('image shape:', img_array.shape)

In [ ]:
def normalize_image(i):
    return (torch.as_tensor(i, dtype=torch.float32) - 0.5) * 2.0   # [0,1] -> [-1,1]

def denormalize_image(i):
    return (i / 2.0) + 0.5                                          # [-1,1] -> [0,1]

def to_display(x):
    x = x.detach().clone()                      # display only; never mutates real values
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def show_image(img, title=None):
    plt.figure(figsize=(3, 3))
    plt.imshow(to_display(img).numpy()); plt.axis('off')
    if title: plt.title(title, fontsize=10)
    plt.show()

def show_strip(frames, titles=None, scale=2.5, cols=6):
    n = len(frames)
    cols = min(cols, n)
    rows = -(-n // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(scale*cols, scale*rows))
    axes = np.array(axes).reshape(-1)
    for k, ax in enumerate(axes):
        if k < n:
            ax.imshow(to_display(frames[k]).numpy())
            if titles: ax.set_title(titles[k], fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.show()

x_0 = normalize_image(img_array)
show_image(x_0, 'x_0  (normalized to [-1, 1])')

---
## Task 1 — the $\beta$ schedule

Use a **linear** schedule. Set these constants exactly:

- `T = 25`
- `beta_0 = 0.0`
- `delta = 0.01`

Then define three tensors, named exactly:

- `beta` — a tensor of length `T + 1` with `beta[t] = beta_0 + delta * t` for `t = 0 .. T`.
  (Index 0 is the "no noise" entry; `beta[0]` should be 0.)
- `alpha` — equal to `1 - beta`.
- `alpha_bar` — the cumulative product of `alpha` along dimension 0.

**torch hints:** build `beta` with a list comprehension inside `torch.tensor([...])`; use
`torch.cumprod(alpha, 0)` for the cumulative product.

The print block below is given — leave it as is. Use it to check your values: you should see
`alpha_bar[T]` around `0.04` and a noise std near `0.99`.

In [ ]:
# ===== YOUR CODE: define T, beta_0, delta, beta, alpha, alpha_bar =====



# ===== GIVEN — sanity prints, do not edit =====
print(f"beta[1]            = {beta[1]:.4f}")
print(f"beta[T]            = {beta[T]:.4f}")
print(f"alpha_bar[T]       = {alpha_bar[T]:.4f}")
print(f"signal scale at T  = {torch.sqrt(alpha_bar[T]):.4f}")
print(f"noise std at T     = {torch.sqrt(1 - alpha_bar[T]):.4f}")

---
## Task 2 — one forward step

Write `forward_sample(x, t)` that performs a **single** noising step:

$$x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{\beta_t}\,z, \qquad z \sim \mathcal{N}(0,1).$$

It should draw fresh noise, scale the input by $\sqrt{\alpha_t}$, add $\sqrt{\beta_t}$
times the noise, and return the result.

**Important:** use the single-step quantities `alpha[t]` and `beta[t]` — **not**
`alpha_bar[t]`. (`alpha_bar` is for jumping straight from $x_0$; here we take one notch at
a time.)

**torch hints:** `torch.randn_like(x)` gives noise shaped like `x`; `torch.sqrt(...)` for
the square roots.

In [ ]:
def forward_sample(x, t):
    # ===== YOUR CODE =====

    pass

---
## Task 3 — one reverse step

Write `reverse_sample(x_0, x_t, t)` that samples $x_{t-1}$ from the reverse posterior
$q(x_{t-1}\mid x_t, x_0)$. Use these formulas (from lecture 2):

$$\tilde\mu_t = \frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,x_0
   + \frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,x_t,
   \qquad
   \tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\,\beta_t.$$

Then return $x_{t-1} = \tilde\mu_t + \sqrt{\tilde\beta_t}\,z$ with fresh
$z \sim \mathcal{N}(0,1)$.

**Special case:** at `t == 1` set the standard deviation to zero (use `torch.tensor(0.0)`),
so the final step is deterministic. Otherwise use $\sqrt{\tilde\beta_t}$.

**torch hints:** you'll index `alpha_bar[t]` and `alpha_bar[t-1]`; use `torch.sqrt(...)`
and `torch.randn_like(x_t)`. Build `mu` as a sum of two scaled terms exactly like the
formula reads.

In [ ]:
def reverse_sample(x_0, x_t, t):
    # ===== YOUR CODE =====
    # 1. pull out abar_t = alpha_bar[t], abar_tm1 = alpha_bar[t-1]
    # 2. build mu from the two-term formula above
    # 3. set sigma = sqrt(beta_tilde) if t > 1 else torch.tensor(0.0)
    # 4. return mu + sigma * noise

    pass

---
## Task 4 — sanity check: one step out, one step back

Take `x_0`, run one forward step to get `x_1`, then one reverse step back. Display all three
side by side. The reversed image should look like the original (the `t=1` reverse step is
deterministic, so it lands exactly on `x_0`).

**What to write:** call `forward_sample` then `reverse_sample`, then
`show_strip([...], ['x_0', 'x_1 (noised)', 'reversed'])`.

In [ ]:
# ===== YOUR CODE =====


---
## Task 5 — forward all the way: watch it dissolve

Starting from `x_0`, apply `forward_sample` for `t = 1, 2, ..., T`, feeding each result into
the next call. Collect every intermediate image in a list called `forward_frames` (start the
list with `x_0` so it has `T + 1` entries). Save the final fully-noised image as `x_T`.

Then display every 5th frame with `show_strip`.

**torch / python hints:** a plain `for t in range(1, T + 1):` loop; append to a list with
`forward_frames.append(x)`. To pick every 5th frame: `[forward_frames[i] for i in range(0, T + 1, 5)]`.

In [ ]:
# ===== YOUR CODE =====
# build forward_frames (starting with x_0), set x_T, then show_strip every 5th frame


---
## Task 6 — reverse all the way: watch it reassemble

Starting from `x_T`, apply `reverse_sample(x_0, x, t)` for `t = T, T-1, ..., 1`, feeding each
result into the next call. Collect the frames in `reverse_frames` (start with `x_T`). Save
the final image as `recovered`.

Display every 5th frame, then show `x_0` next to `recovered` and print the recovery RMSE.

**hints:** loop with `for t in range(T, 0, -1):`. The RMSE is
`torch.sqrt(((recovered - x_0) ** 2).mean())`. A correct implementation prints an RMSE very
close to 0.

In [ ]:
# ===== YOUR CODE =====
# build reverse_frames (starting with x_T), set recovered, show_strip every 5th frame



# ===== GIVEN — final comparison, do not edit =====
show_strip([x_0, recovered], ['original x_0', 'recovered'])
rmse = torch.sqrt(((recovered - x_0) ** 2).mean())
print(f'recovery RMSE: {rmse:.6f}')

---
## Task 7 — reflection (write a short answer)

Your reverse process recovered the image almost perfectly. But it used the true `x_0` inside
every reverse step.

1. Look at the formula for $\tilde\mu_t$. Which term would you be unable to compute if you
   did **not** know `x_0`?
2. In a real image generator you start from pure noise and have no `x_0`. What would have to
   replace that term, and how might you obtain it?
3. Try changing `delta` to `0.03` and re-running. Why does the final frame get so much
   noisier from such a small change? (Hint: think about how the surviving signal
   $\sqrt{\bar\alpha_T}$ depends on the whole schedule.)

*Write your answers here.*